# Clinical Trial Randomization Simulator – Solution Notebook

**Health Industry Application – Clinical Research Education**

Adapted from the Rock Paper Scissors extended project.
This tool teaches why proper randomization is essential in clinical trials by letting you
simulate different allocation strategies and observe balance vs imbalance over many trials.

---
## Flowchart of the Desired Outcome

In [1]:
from IPython.display import Image, display
display(Image(filename='clinical_trial_flowchart.png', width=850))

## 1. Imports and Trial Parameters

We define the basic building blocks of a two-arm trial: Treatment vs Control.

In [2]:
import random
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Core trial settings (can be changed later in the simulation section)
N_PATIENTS = 100          # total patients to randomize
ARMS = ['Treatment', 'Control']
ALLOCATION_RATIO = 0.5    # target proportion in Treatment arm

print(f'Trial size: {N_PATIENTS} patients')
print(f'Arms: {ARMS}')
print(f'Target allocation to Treatment: {ALLOCATION_RATIO*100:.0f}%')

Trial size: 100 patients
Arms: ['Treatment', 'Control']
Target allocation to Treatment: 50%


## 2. Randomization Methods (Primary + Alternates)

### Primary: Simple randomization (independent coin flip for each patient)
### Alternate 1: Permuted-block randomization (guarantees balance within blocks)
### Alternate 2: Biased / deterministic strategies (for teaching what goes wrong)

In [3]:
def simple_randomization(n_patients, p_treatment=0.5, seed=None):
    """Independent Bernoulli trial for each patient."""
    if seed is not None:
        random.seed(seed)
    return ['Treatment' if random.random() < p_treatment else 'Control'
            for _ in range(n_patients)]

def block_randomization(n_patients, block_size=4, p_treatment=0.5, seed=None):
    """Permuted-block randomization. block_size should be even for 1:1."""
    if seed is not None:
        random.seed(seed)
    assignments = []
    n_treat_per_block = int(block_size * p_treatment)
    while len(assignments) < n_patients:
        block = (['Treatment'] * n_treat_per_block +
                 ['Control'] * (block_size - n_treat_per_block))
        random.shuffle(block)
        assignments.extend(block)
    return assignments[:n_patients]

def biased_always_treatment(n_patients):
    """Deterministic – every patient to Treatment (extreme selection bias)."""
    return ['Treatment'] * n_patients

def biased_sequential(n_patients):
    """Alternate T/C/T/C... (predictable, not random)."""
    return ['Treatment' if i % 2 == 0 else 'Control' for i in range(n_patients)]

# Quick demo
print('Simple (seed=42):', simple_randomization(10, seed=42))
print('Block  (seed=42):', block_randomization(10, block_size=4, seed=42))
print('Always Treatment:', biased_always_treatment(6))
print('Sequential:      ', biased_sequential(6))

Simple (seed=42): ['Control', 'Control', 'Treatment', 'Treatment', 'Control', 'Treatment', 'Treatment', 'Control', 'Control', 'Control']
Block  (seed=42): ['Control', 'Treatment', 'Treatment', 'Control', 'Control', 'Treatment', 'Treatment', 'Control', 'Treatment', 'Control']
Always Treatment: ['Treatment', 'Treatment', 'Treatment', 'Treatment', 'Treatment', 'Treatment']
Sequential:       ['Treatment', 'Control', 'Treatment', 'Control', 'Treatment', 'Control']


## 3. Single-Trial Summary Function

Given a list of assignments, compute counts, percentage in Treatment, and absolute imbalance.

In [4]:
def summarize_trial(assignments):
    """Return dict with counts, pct_treatment, and imbalance."""
    counts = Counter(assignments)
    n = len(assignments)
    n_treat = counts.get('Treatment', 0)
    pct = n_treat / n if n else 0
    imbalance = abs(n_treat - (n - n_treat))   # |T - C|
    return {
        'n': n,
        'Treatment': n_treat,
        'Control': counts.get('Control', 0),
        'pct_treatment': pct,
        'imbalance': imbalance
    }

# Demo on the methods above
for name, func in [('Simple', lambda: simple_randomization(100, seed=1)),
                   ('Block',  lambda: block_randomization(100, seed=1)),
                   ('Always T', biased_always_treatment),
                   ('Sequential', biased_sequential)]:
    s = summarize_trial(func(100) if name != 'Always T' and name != 'Sequential' else func(100))
    print(f"{name:12s} → T={s['Treatment']:3d}  C={s['Control']:3d}  "
          f"pct={s['pct_treatment']*100:5.1f}%  imbalance={s['imbalance']}")

Simple       → T= 52  C= 48  pct= 52.0%  imbalance=4
Block        → T= 50  C= 50  pct= 50.0%  imbalance=0
Always T     → T=100  C=  0  pct=100.0%  imbalance=100
Sequential   → T= 50  C= 50  pct= 50.0%  imbalance=0


## 4. Monte-Carlo Simulation Engine

Run many independent trials and collect the distribution of the Treatment percentage
(or imbalance). This is the heart of the educational demonstration.

In [5]:
def run_monte_carlo(method_func, n_patients=100, n_trials=2000, seed=42):
    """
    method_func: a function that takes n_patients and returns a list of assignments.
    Returns list of pct_treatment from each simulated trial.
    """
    random.seed(seed)
    pcts = []
    for i in range(n_trials):
        # re-seed per trial for independence while keeping overall reproducibility
        assignments = method_func(n_patients)
        s = summarize_trial(assignments)
        pcts.append(s['pct_treatment'])
    return pcts

# Example: simple randomization
pcts_simple = run_monte_carlo(lambda n: simple_randomization(n), n_patients=100, n_trials=2000)
print(f'Simple randomization – mean % Treatment: {np.mean(pcts_simple)*100:.1f}%')
print(f'                       std:               {np.std(pcts_simple)*100:.1f}%')
print(f'                       min–max:           {min(pcts_simple)*100:.1f}% – {max(pcts_simple)*100:.1f}%')

Simple randomization – mean % Treatment: 50.0%
                       std:               5.0%
                       min–max:           34.0% – 66.0%


## 5. Visualization of Imbalance Distributions

Histograms show how tightly the % Treatment clusters around the target (50%).
Fair methods produce a nice bell shape; biased methods produce spikes far from 50%.

In [6]:
# Generate data for four methods
N_TRIALS = 2000
N_PAT = 100

pcts = {
    'Simple Random': run_monte_carlo(lambda n: simple_randomization(n), N_PAT, N_TRIALS, seed=10),
    'Block (size=4)': run_monte_carlo(lambda n: block_randomization(n, block_size=4), N_PAT, N_TRIALS, seed=10),
    'Always Treatment': run_monte_carlo(lambda n: biased_always_treatment(n), N_PAT, N_TRIALS, seed=10),
    'Sequential': run_monte_carlo(lambda n: biased_sequential(n), N_PAT, N_TRIALS, seed=10),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()
colors = ['#27AE60', '#2980B9', '#E74C3C', '#F39C12']

for ax, (name, data), color in zip(axes, pcts.items(), colors):
    ax.hist([p*100 for p in data], bins=20, color=color, edgecolor='black', alpha=0.85)
    ax.axvline(50, color='black', linestyle='--', linewidth=1.5, label='Target 50%')
    ax.set_title(name)
    ax.set_xlabel('% assigned to Treatment')
    ax.set_ylabel('Number of simulated trials')
    ax.set_xlim(0, 100)
    ax.legend(fontsize=8)

plt.suptitle(f'Distribution of Treatment Allocation\n({N_TRIALS} trials × {N_PAT} patients)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('clinical_trial_simulation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → clinical_trial_simulation.png')

Saved → clinical_trial_simulation.png


## 6. Strategy Comparison Table (numeric summary)

In [7]:
print(f'{"Method":20s}  {"Mean %T":>8s}  {"Std":>6s}  {"Min":>6s}  {"Max":>6s}  {"P(|imb|>10)":>12s}')
print('-'*70)
for name, data in pcts.items():
    arr = np.array(data) * 100
    imb = np.abs(arr - 50)
    p_bad = np.mean(imb > 10) * 100
    print(f'{name:20s}  {arr.mean():8.1f}  {arr.std():6.1f}  {arr.min():6.1f}  {arr.max():6.1f}  {p_bad:10.1f}%')

Method                 Mean %T     Std     Min     Max  P(|imb|>10)
----------------------------------------------------------------------
Simple Random             50.0     5.0    34.0    66.0        5.0%
Block (size=4)            50.0     0.0    50.0    50.0        0.0%
Always Treatment         100.0     0.0   100.0   100.0      100.0%
Sequential                50.0     0.0    50.0    50.0        0.0%


## 7. More Practice Exercises

### Practice A – 2:1 Allocation Ratio
Modify simple and block randomization to target 2/3 Treatment (common in some oncology trials).
Re-run a small Monte-Carlo and check the mean lands near 66.7%.

In [8]:
# 2:1 allocation demo
pcts_2to1 = run_monte_carlo(lambda n: simple_randomization(n, p_treatment=2/3), n_patients=90, n_trials=1000, seed=7)
print(f'2:1 Simple – mean % Treatment: {np.mean(pcts_2to1)*100:.1f}%  (target ≈ 66.7%)')

pcts_block_2to1 = run_monte_carlo(lambda n: block_randomization(n, block_size=6, p_treatment=2/3),
                                  n_patients=90, n_trials=1000, seed=7)
print(f'2:1 Block  – mean % Treatment: {np.mean(pcts_block_2to1)*100:.1f}%  (target ≈ 66.7%)')

2:1 Simple – mean % Treatment: 66.7%  (target ≈ 66.7%)
2:1 Block  – mean % Treatment: 66.7%  (target ≈ 66.7%)


### Practice B – Detecting Chance Imbalance
Even fair simple randomization can produce large imbalances by chance, especially with small N.
Explore how the probability of |imbalance| > 10 changes with sample size.

In [9]:
print('Probability of |%T - 50| > 10 under simple randomization:')
for n in [20, 50, 100, 200, 500]:
    pcts = run_monte_carlo(lambda n_: simple_randomization(n_), n_patients=n, n_trials=3000, seed=99)
    p = np.mean(np.abs(np.array(pcts)*100 - 50) > 10) * 100
    print(f'  N = {n:3d}  →  {p:5.1f}%')

Probability of |%T - 50| > 10 under simple randomization:
  N =  20  →   50.3%
  N =  50  →   20.2%
  N = 100  →    5.1%
  N = 200  →    0.5%
  N = 500  →    0.0%


## 8. Tunable Simulation Section

**Modify the parameters below and re-run** to explore different scenarios.

In [10]:
# ========== TUNABLE PARAMETERS ==========
N_PATIENTS   = 80
N_TRIALS     = 1500
METHOD       = 'simple'   # 'simple' | 'block' | 'always_t' | 'sequential'
BLOCK_SIZE   = 4
P_TREATMENT  = 0.5
SEED         = 123
# ========================================

def get_method(method):
    if method == 'simple':
        return lambda n: simple_randomization(n, p_treatment=P_TREATMENT)
    elif method == 'block':
        return lambda n: block_randomization(n, block_size=BLOCK_SIZE, p_treatment=P_TREATMENT)
    elif method == 'always_t':
        return biased_always_treatment
    elif method == 'sequential':
        return biased_sequential
    else:
        raise ValueError('Unknown method')

pcts_tune = run_monte_carlo(get_method(METHOD), n_patients=N_PATIENTS, n_trials=N_TRIALS, seed=SEED)
arr = np.array(pcts_tune) * 100
print(f'Method={METHOD} | N={N_PATIENTS} | trials={N_TRIALS}')
print(f'Mean % Treatment : {arr.mean():.1f}%')
print(f'Std              : {arr.std():.1f}%')
print(f'Range            : {arr.min():.1f}% – {arr.max():.1f}%')
print(f'P(|dev| > 10)    : {np.mean(np.abs(arr-50)>10)*100:.1f}%')

Method=simple | N=80 | trials=1500
Mean % Treatment : 50.0%
Std              : 5.6%
Range            : 32.5% – 67.5%
P(|dev| > 10)    : 8.7%


## 9. Key Educational Insights (from the original RPS exploration questions)

1. Changing a fair random process into a deterministic rule (like always assigning Treatment)
   destroys balance and introduces selection bias – exactly why clinical trials forbid it.

2. Even fair simple randomization can, by pure chance, produce noticeable imbalance when N is small.
   This is why many modern trials use block or stratified randomization.

3. The simulation approach (Monte-Carlo) is the same technique used by statisticians to
   evaluate the operating characteristics of a randomization scheme before a real trial begins.

## Key Takeaways

- Proper randomization (simple or blocked) keeps the expected allocation at the design target and limits chance imbalance.
- Deterministic or biased strategies produce severe, systematic imbalance – the educational analogue of selection bias.
- Sample size matters: larger N makes large chance imbalances rare under simple randomization.
- Monte-Carlo simulation is a practical way to stress-test a randomization plan before locking the protocol.
- The same programming patterns used in a simple game (random choice, strategy comparison, simulation loops)
  transfer directly to rigorous clinical-research education tools.